# Ingesting fact data into bronze layer
Loading all 90 daily enrollment CSV files into a single bronze delta table.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
import pyspark.sql.functions as F
catalog_name = 'edutrack'

In [0]:
enrollment_schema = StructType([
    StructField('dt',               StringType(), True),
    StructField('enrollment_id',    StringType(), False),
    StructField('student_id',       StringType(), True),
    StructField('course_id',        StringType(), True),
    StructField('instructor_id',    StringType(), True),
    StructField('semester',         StringType(), True),
    StructField('marks_obtained',   StringType(), True),
    StructField('max_marks',        StringType(), True),
    StructField('attendance_pct',   StringType(), True),
    StructField('grade',            StringType(), True),
    StructField('assignment_score', StringType(), True),
    StructField('status',           StringType(), True),
])

In [0]:
raw_path = '/Volumes/edutrack/source_data/raw/enrollments/landing/*.csv'
df = (spark.read.option('header','true').option('delimiter',',')
      .schema(enrollment_schema)
      .csv(raw_path)
      .withColumn('file_name', F.col('_metadata.file_path'))
      .withColumn('ingest_timestamp', F.current_timestamp()))
print(f'Total rows ingested (inc. duplicates): {df.count()}')
display(df.limit(5))

In [0]:
df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('edutrack.bronze.brz_enrollments')
print('brz_enrollments:', spark.table('edutrack.bronze.brz_enrollments').count(),'rows')